# AI Layer Developer Guide

**Closed-Loop Intent-Based Network System**

*Developer documentation for the Reinforcement Learning traffic optimization component.*


### 1 Purpose

The AI Layer implements a Deep Q-Network (DQN) agent that optimizes SDN traffic routing and rate limiting decisions. It learns to balance network load across links while minimizing packet loss.


### 2 System Context

```
┌─────────────────────────────────────────────────────────────────┐
│                        External Systems                         │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────────────┐ │
│  │   Mininet   │    │ Ryu SDN     │    │    OpenFlow Switch   │ │
│  │ (Traffic)   │    │ Controller  │    │    (DPID: 00...01)  │ │
│  └──────┬──────┘    └──────┬──────┘    └──────────┬──────────┘ │
└─────────┼──────────────────┼─────────────────────┼─────────────┘
                             │                     │
                             │ REST API            │
                             │ (localhost:8080)    │
                             ▼                     │
┌──────────────────────────────────────────────────────────────┐
│                      AI Layer (This Project)                │
│                                                             │
│  ┌──────-──────┐    ┌─────────────┐    ┌─────────────────┐    │
│  │  Telemetry  │───▶│    State    │───▶│   DQN Agent     │    │
│  │   Parser    │    │   Builder   │    │  (6→64→64→4)    │    │
│  └─────────────┘    └─────────────┘    └────────┬────────┘    │
│                                                  │             │
│         ┌────────────────────────────────────────┘             │
│         ▼                                                      │
│  ┌─────────────────┐    ┌─────────────┐                        │
│  │    Action       │───▶│  REST Client│─────────────────────▶  │
│  │   Translator    │    │  (requests) │                        │
│  └─────────────────┘    └─────────────┘                        │
│                                                                │
└────────────────────────────────────────────────────────────────┘
```


## 2. Technology Stack

| Component | Technology | Version | Purpose |
|-----------|------------|---------|---------|
| RL Framework | PyTorch | ≥2.0 | Neural network, training loop |
| API Client | requests | ≥2.28 | HTTP communication with Ryu |
| Monitoring | TensorBoard | ≥2.12 | Training visualization |
| Configuration | JSON | — | Environment parameters |
| Environment Wrapper | gym | ≥0.26 | RL environment interface |

## Data Flow

```
Ryu REST API → Telemetry Parser (pluggable) → State Vector Builder → Normalization → RL Agent
```

## 3. Reward Function

| Component | Weight | Description |
|-----------|--------|-------------|
| Throughput Bonus | +1.5 | Reward for maintaining higher normalized throughput |
| Latency Penalty | -2.5 | Penalty for high latency inside the 10-80 ms operating bounds |
| Packet Loss Penalty | -3.0 | Heavy penalty for packet loss (0-5% range tracking) |
| Utilization Penalty | -1.2 | Penalty for sustained high average path utilization |
| Congestion Penalty | -2.0 | Extra penalty if any path util crosses 0.9 threshold |
| Failover Penalty | -0.2 | Small tracking cost to discourage unnecessary backup-path residency |


## 4.DQN Agent Architecture

```
6 neurons (state dimension)

Hidden Layer 1: 64 neurons, ReLU activation
Hidden Layer 2: 64 neurons, ReLU activation

Output Layer:   5 neurons (action Q-values)
```

### Hyperparameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| `learning_rate` | 0.001 | Adam optimizer |
| `gamma` | 0.99 | Discount factor |
| `epsilon_start` | 1.0 | Initial exploration rate |
| `epsilon_end` | 0.01 | Minimum exploration rate |
| `epsilon_decay` | 0.995 | Per-episode decay |
| `batch_size` | 64 | Training batch size |
| `target_update_freq` | 100 | Steps between target network updates |

### Replay Buffer

| Parameter | Value |
|-----------|-------|
| `capacity` | 10,000 transitions |
| `min_size_for_training` | 1,000 transitions |

### Loss Function

- **MSE** (Mean Squared Error) for Q-value regression

---

## 5. Training Phase

### Training Step Execution Cycle

| Phase | Description |
|---|---|
| **1. Observation** | The Gymnasium wrapper calls Ryu APIs (`/links/utilization`, `/latency/src/dst`) and normalizes the data into a 6D mathematical state vector. |
| **2. Action Selection** | The DQN uses its $\epsilon$-greedy policy to either guess a random action or predict the optimal action (e.g., `update_queue`) based on the current state. |
| **3. Execution** | The `ActionTranslator` fires the corresponding POST operation to the live Ryu controller to physically change routing or switch QoS. |
| **4. Delay** | Wait `stabilization_delay_seconds` (e.g., 1s-2s) to allow Mininet traffic and OpenFlow rules to stabilize. |
| **5. Measurement** | Take a new observation to see how the network reacted to the action. |
| **6. Reward & Buffer** | Calculate the mathematical reward (+ or - penalty) and store the `[Old State, Action, Reward, New State]` block into the replay memory. |
| **7. Learn** | If the memory buffer is full enough, grab a random batch of 64 past experiences and tweak the Neural Network's internal weights via PyTorch. |

### Workflow
- **Inference Mode:** Exploration is completely disabled ($\epsilon = 0$). The neural network acts purely on its trained knowledge.
- **Baselines:** The agent's performance is strictly compared against naive baseline policies:
  - `random`: Picks actions randomly.
  - `do_nothing`: Always leaves the network at its baseline default.

### Execution & Outputs
- **Command:** `python evaluate.py --config prod.json --model-path models/dqn_model.pth`
- **Outputs:** A comprehensive report (`logs/evaluation_metrics.json`) detailing final average rewards, action distributions, success rates, and comparative proxy metrics for latency and packet loss.